In [ ]:
import pandas as pd
from functions.eval import *
from transformers import pipeline, AutoTokenizer
import torch
from tqdm.notebook import tqdm
tqdm.pandas()

In [2]:
# model_pred_col = "Camelbert-MSA"
# model_name = "CAMeL-Lab/bert-base-arabic-camelbert-msa-sentiment"
model_name = "PRAli22/AraBert-Arabic-Sentiment-Analysis"
model_pred_col = "AraBert"

In [3]:
xai_exec = pd.read_csv("data/xai_exec/xai_exec_" + model_pred_col + ".csv")

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(model_name)
pipe = pipeline("text-classification", model=model_name, top_k=None, device=device, 
                truncation=True, max_length=512)

Device set to use cuda


In [5]:
xai_exec.columns

Index(['text', 'AraBert', 'LIME', 'SHAP', 'IG', 'DeepLIFT',
       'EnsembleXAI_LIME_SHAP_IG_DL_mean',
       'EnsembleXAI_LIME_SHAP_IG_DL_median', 'EnsembleXAI_LIME_SHAP_IG_mean',
       'EnsembleXAI_LIME_SHAP_IG_median', 'EnsembleXAI_LIME_SHAP_mean',
       'EnsembleXAI_LIME_SHAP_median'],
      dtype='str')

In [6]:
lime_eval = xai_exec[["text", model_pred_col, "LIME"]]
shap_eval = xai_exec[["text", model_pred_col, "SHAP"]]
ig_eval = xai_exec[["text", model_pred_col, "IG"]]
dl_eval = xai_exec[["text", model_pred_col, "DeepLIFT"]]
ensemble_eval = xai_exec[["text", model_pred_col, "EnsembleXAI_LIME_SHAP_IG_DL_mean", 
    "EnsembleXAI_LIME_SHAP_IG_DL_median", "EnsembleXAI_LIME_SHAP_IG_mean", "EnsembleXAI_LIME_SHAP_IG_median", 
    "EnsembleXAI_LIME_SHAP_mean", "EnsembleXAI_LIME_SHAP_median"]]

In [7]:
prediction_cache = {}

In [8]:
lime_eval["LIME_comprehensiveness"] = lime_eval.progress_apply(lambda row: comprehensivness(tokenizer, pipe, row[model_pred_col], eval(row["LIME"]), class_proba(pipe, row["text"]), prediction_cache), axis=1)
lime_eval["LIME_sufficiency"] = lime_eval.progress_apply(lambda row: sufficiency(tokenizer, pipe, row[model_pred_col], eval(row["LIME"]), class_proba(pipe, row["text"]), prediction_cache), axis=1)
lime_eval["LIME_corr_loo"] = lime_eval.progress_apply(lambda row: correlation_leave_one_out(tokenizer, pipe, row[model_pred_col], eval(row["LIME"]), class_proba(pipe, row["text"]), prediction_cache), axis=1)
lime_eval["LIME_ins_AUC"] = lime_eval.progress_apply(lambda row: insertion_auc(tokenizer, pipe, row[model_pred_col], eval(row["LIME"]), prediction_cache), axis=1)
lime_eval["LIME_del_AUC"] = lime_eval.progress_apply(lambda row: deletion_auc(tokenizer, pipe, row[model_pred_col], eval(row["LIME"]), prediction_cache), axis=1)

  0%|          | 0/3033 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

In [9]:
lime_eval["LIME_combined"] = lime_eval.apply(lambda row: combined_metric(row["LIME_comprehensiveness"], row["LIME_sufficiency"], row["LIME_corr_loo"], row["LIME_ins_AUC"], row["LIME_del_AUC"]), axis=1)

In [10]:
lime_eval.to_csv("data/xai_eval/xai_eval_lime_" + model_pred_col + ".csv", index=False)

In [11]:
shap_eval["SHAP_comprehensiveness"] = shap_eval.progress_apply(lambda row: comprehensivness(tokenizer, pipe, row[model_pred_col], eval(row["SHAP"]), class_proba(pipe, row["text"]), prediction_cache), axis=1)
shap_eval["SHAP_sufficiency"] = shap_eval.progress_apply(lambda row: sufficiency(tokenizer, pipe, row[model_pred_col], eval(row["SHAP"]), class_proba(pipe, row["text"]), prediction_cache), axis=1)
shap_eval["SHAP_corr_loo"] = shap_eval.progress_apply(lambda row: correlation_leave_one_out(tokenizer, pipe, row[model_pred_col], eval(row["SHAP"]), class_proba(pipe, row["text"]), prediction_cache), axis=1)
shap_eval["SHAP_ins_AUC"] = shap_eval.progress_apply(lambda row: insertion_auc(tokenizer, pipe, row[model_pred_col], eval(row["SHAP"]), prediction_cache), axis=1)
shap_eval["SHAP_del_AUC"] = shap_eval.progress_apply(lambda row: deletion_auc(tokenizer, pipe, row[model_pred_col], eval(row["SHAP"]), prediction_cache), axis=1)

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

In [12]:
shap_eval["SHAP_combined"] = shap_eval.apply(lambda row: combined_metric(row["SHAP_comprehensiveness"], row["SHAP_sufficiency"], row["SHAP_corr_loo"], row["SHAP_ins_AUC"], row["SHAP_del_AUC"]), axis=1)

In [13]:
shap_eval.to_csv("data/xai_eval/xai_eval_shap_" + model_pred_col + ".csv", index=False)

In [14]:
ig_eval["IG_comprehensiveness"] = ig_eval.progress_apply(lambda row: comprehensivness(tokenizer, pipe, row[model_pred_col], eval(row["IG"]), class_proba(pipe, row["text"]), prediction_cache), axis=1)
ig_eval["IG_sufficiency"] = ig_eval.progress_apply(lambda row: sufficiency(tokenizer, pipe, row[model_pred_col], eval(row["IG"]), class_proba(pipe, row["text"]), prediction_cache), axis=1)
ig_eval["IG_corr_loo"] = ig_eval.progress_apply(lambda row: correlation_leave_one_out(tokenizer, pipe, row[model_pred_col], eval(row["IG"]), class_proba(pipe, row["text"]), prediction_cache), axis=1)
ig_eval["IG_ins_AUC"] = ig_eval.progress_apply(lambda row: insertion_auc(tokenizer, pipe, row[model_pred_col], eval(row["IG"]), prediction_cache), axis=1)
ig_eval["IG_del_AUC"] = ig_eval.progress_apply(lambda row: deletion_auc(tokenizer, pipe, row[model_pred_col], eval(row["IG"]), prediction_cache), axis=1)

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

In [15]:
ig_eval["IG_combined"] = ig_eval.apply(lambda row: combined_metric(row["IG_comprehensiveness"], row["IG_sufficiency"], row["IG_corr_loo"], row["IG_ins_AUC"], row["IG_del_AUC"]), axis=1)

In [16]:
ig_eval.to_csv("data/xai_eval/xai_eval_ig_" + model_pred_col + ".csv", index=False)

In [17]:
dl_eval["DeepLIFT_comprehensiveness"] = dl_eval.progress_apply(lambda row: comprehensivness(tokenizer, pipe, row[model_pred_col], eval(row["DeepLIFT"]), class_proba(pipe, row["text"]), prediction_cache), axis=1)
dl_eval["DeepLIFT_sufficiency"] = dl_eval.progress_apply(lambda row: sufficiency(tokenizer, pipe, row[model_pred_col], eval(row["DeepLIFT"]), class_proba(pipe, row["text"]), prediction_cache), axis=1)
dl_eval["DeepLIFT_corr_loo"] = dl_eval.progress_apply(lambda row: correlation_leave_one_out(tokenizer, pipe, row[model_pred_col], eval(row["DeepLIFT"]), class_proba(pipe, row["text"]), prediction_cache), axis=1)
dl_eval["DeepLIFT_ins_AUC"] = dl_eval.progress_apply(lambda row: insertion_auc(tokenizer, pipe, row[model_pred_col], eval(row["DeepLIFT"]), prediction_cache), axis=1)
dl_eval["DeepLIFT_del_AUC"] = dl_eval.progress_apply(lambda row: deletion_auc(tokenizer, pipe, row[model_pred_col], eval(row["DeepLIFT"]), prediction_cache), axis=1)

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

In [18]:
dl_eval["DeepLIFT_combined"] = dl_eval.apply(lambda row: combined_metric(row["DeepLIFT_comprehensiveness"], row["DeepLIFT_sufficiency"], row["DeepLIFT_corr_loo"], row["DeepLIFT_ins_AUC"], row["DeepLIFT_del_AUC"]), axis=1)

In [19]:
dl_eval.to_csv("data/xai_eval/xai_eval_dl_" + model_pred_col + ".csv", index=False)

In [20]:
ensemble_eval["EXAI_LIME_SHAP_IG_DL_mean_comprehensiveness"] = ensemble_eval.progress_apply(lambda row: comprehensivness(tokenizer, pipe, row[model_pred_col], eval(row["EnsembleXAI_LIME_SHAP_IG_DL_mean"]), class_proba(pipe, row["text"]), prediction_cache), axis=1)
ensemble_eval["EXAI_LIME_SHAP_IG_DL_mean_sufficiency"] = ensemble_eval.progress_apply(lambda row: sufficiency(tokenizer, pipe, row[model_pred_col], eval(row["EnsembleXAI_LIME_SHAP_IG_DL_mean"]), class_proba(pipe, row["text"]), prediction_cache), axis=1)
ensemble_eval["EXAI_LIME_SHAP_IG_DL_mean_corr_loo"] = ensemble_eval.progress_apply(lambda row: correlation_leave_one_out(tokenizer, pipe, row[model_pred_col], eval(row["EnsembleXAI_LIME_SHAP_IG_DL_mean"]), class_proba(pipe, row["text"]), prediction_cache), axis=1)
ensemble_eval["EXAI_LIME_SHAP_IG_DL_mean_ins_AUC"] = ensemble_eval.progress_apply(lambda row: insertion_auc(tokenizer, pipe, row[model_pred_col], eval(row["EnsembleXAI_LIME_SHAP_IG_DL_mean"]), prediction_cache), axis=1)
ensemble_eval["EXAI_LIME_SHAP_IG_DL_mean_del_AUC"] = ensemble_eval.progress_apply(lambda row: deletion_auc(tokenizer, pipe, row[model_pred_col], eval(row["EnsembleXAI_LIME_SHAP_IG_DL_mean"]), prediction_cache), axis=1)

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

In [21]:
ensemble_eval["EXAI_LIME_SHAP_IG_DL_mean_combined"] = ensemble_eval.apply(lambda row: combined_metric(row["EXAI_LIME_SHAP_IG_DL_mean_comprehensiveness"], row["EXAI_LIME_SHAP_IG_DL_mean_sufficiency"], row["EXAI_LIME_SHAP_IG_DL_mean_corr_loo"], row["EXAI_LIME_SHAP_IG_DL_mean_ins_AUC"], row["EXAI_LIME_SHAP_IG_DL_mean_del_AUC"]), axis=1)

In [22]:
ensemble_eval["EXAI_LIME_SHAP_IG_DL_median_comprehensiveness"] = ensemble_eval.progress_apply(lambda row: comprehensivness(tokenizer, pipe, row[model_pred_col], eval(row["EnsembleXAI_LIME_SHAP_IG_DL_median"]), class_proba(pipe, row["text"]), prediction_cache), axis=1)
ensemble_eval["EXAI_LIME_SHAP_IG_DL_median_sufficiency"] = ensemble_eval.progress_apply(lambda row: sufficiency(tokenizer, pipe, row[model_pred_col], eval(row["EnsembleXAI_LIME_SHAP_IG_DL_median"]), class_proba(pipe, row["text"]), prediction_cache), axis=1)
ensemble_eval["EXAI_LIME_SHAP_IG_DL_median_corr_loo"] = ensemble_eval.progress_apply(lambda row: correlation_leave_one_out(tokenizer, pipe, row[model_pred_col], eval(row["EnsembleXAI_LIME_SHAP_IG_DL_median"]), class_proba(pipe, row["text"]), prediction_cache), axis=1)
ensemble_eval["EXAI_LIME_SHAP_IG_DL_median_ins_AUC"] = ensemble_eval.progress_apply(lambda row: insertion_auc(tokenizer, pipe, row[model_pred_col], eval(row["EnsembleXAI_LIME_SHAP_IG_DL_median"]), prediction_cache), axis=1)
ensemble_eval["EXAI_LIME_SHAP_IG_DL_median_del_AUC"] = ensemble_eval.progress_apply(lambda row: deletion_auc(tokenizer, pipe, row[model_pred_col], eval(row["EnsembleXAI_LIME_SHAP_IG_DL_median"]), prediction_cache), axis=1)

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

In [23]:
ensemble_eval["EXAI_LIME_SHAP_IG_DL_median_combined"] = ensemble_eval.apply(lambda row: combined_metric(row["EXAI_LIME_SHAP_IG_DL_median_comprehensiveness"], row["EXAI_LIME_SHAP_IG_DL_median_sufficiency"], row["EXAI_LIME_SHAP_IG_DL_median_corr_loo"], row["EXAI_LIME_SHAP_IG_DL_median_ins_AUC"], row["EXAI_LIME_SHAP_IG_DL_median_del_AUC"]), axis=1)

In [24]:
ensemble_eval["EXAI_LIME_SHAP_IG_mean_comprehensiveness"] = ensemble_eval.progress_apply(lambda row: comprehensivness(tokenizer, pipe, row[model_pred_col], eval(row["EnsembleXAI_LIME_SHAP_IG_mean"]), class_proba(pipe, row["text"]), prediction_cache), axis=1)
ensemble_eval["EXAI_LIME_SHAP_IG_mean_sufficiency"] = ensemble_eval.progress_apply(lambda row: sufficiency(tokenizer, pipe, row[model_pred_col], eval(row["EnsembleXAI_LIME_SHAP_IG_mean"]), class_proba(pipe, row["text"]), prediction_cache), axis=1)
ensemble_eval["EXAI_LIME_SHAP_IG_mean_corr_loo"] = ensemble_eval.progress_apply(lambda row: correlation_leave_one_out(tokenizer, pipe, row[model_pred_col], eval(row["EnsembleXAI_LIME_SHAP_IG_mean"]), class_proba(pipe, row["text"]), prediction_cache), axis=1)
ensemble_eval["EXAI_LIME_SHAP_IG_mean_ins_AUC"] = ensemble_eval.progress_apply(lambda row: insertion_auc(tokenizer, pipe, row[model_pred_col], eval(row["EnsembleXAI_LIME_SHAP_IG_mean"]), prediction_cache), axis=1)
ensemble_eval["EXAI_LIME_SHAP_IG_mean_del_AUC"] = ensemble_eval.progress_apply(lambda row: deletion_auc(tokenizer, pipe, row[model_pred_col], eval(row["EnsembleXAI_LIME_SHAP_IG_mean"]), prediction_cache), axis=1)

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

In [25]:
ensemble_eval["EXAI_LIME_SHAP_IG_mean_combined"] = ensemble_eval.apply(lambda row: combined_metric(row["EXAI_LIME_SHAP_IG_mean_comprehensiveness"], row["EXAI_LIME_SHAP_IG_mean_sufficiency"], row["EXAI_LIME_SHAP_IG_mean_corr_loo"], row["EXAI_LIME_SHAP_IG_mean_ins_AUC"], row["EXAI_LIME_SHAP_IG_mean_del_AUC"]), axis=1)

In [26]:
ensemble_eval["EXAI_LIME_SHAP_IG_median_comprehensiveness"] = ensemble_eval.progress_apply(lambda row: comprehensivness(tokenizer, pipe, row[model_pred_col], eval(row["EnsembleXAI_LIME_SHAP_IG_median"]), class_proba(pipe, row["text"]), prediction_cache), axis=1)
ensemble_eval["EXAI_LIME_SHAP_IG_median_sufficiency"] = ensemble_eval.progress_apply(lambda row: sufficiency(tokenizer, pipe, row[model_pred_col], eval(row["EnsembleXAI_LIME_SHAP_IG_median"]), class_proba(pipe, row["text"]), prediction_cache), axis=1)
ensemble_eval["EXAI_LIME_SHAP_IG_median_corr_loo"] = ensemble_eval.progress_apply(lambda row: correlation_leave_one_out(tokenizer, pipe, row[model_pred_col], eval(row["EnsembleXAI_LIME_SHAP_IG_median"]), class_proba(pipe, row["text"]), prediction_cache), axis=1)
ensemble_eval["EXAI_LIME_SHAP_IG_median_ins_AUC"] = ensemble_eval.progress_apply(lambda row: insertion_auc(tokenizer, pipe, row[model_pred_col], eval(row["EnsembleXAI_LIME_SHAP_IG_median"]), prediction_cache), axis=1)
ensemble_eval["EXAI_LIME_SHAP_IG_median_del_AUC"] = ensemble_eval.progress_apply(lambda row: deletion_auc(tokenizer, pipe, row[model_pred_col], eval(row["EnsembleXAI_LIME_SHAP_IG_median"]), prediction_cache), axis=1)

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

In [27]:
ensemble_eval["EXAI_LIME_SHAP_IG_median_combined"] = ensemble_eval.apply(lambda row: combined_metric(row["EXAI_LIME_SHAP_IG_median_comprehensiveness"], row["EXAI_LIME_SHAP_IG_median_sufficiency"], row["EXAI_LIME_SHAP_IG_median_corr_loo"], row["EXAI_LIME_SHAP_IG_median_ins_AUC"], row["EXAI_LIME_SHAP_IG_median_del_AUC"]), axis=1)

In [28]:
ensemble_eval["EXAI_LIME_SHAP_mean_comprehensiveness"] = ensemble_eval.progress_apply(lambda row: comprehensivness(tokenizer, pipe, row[model_pred_col], eval(row["EnsembleXAI_LIME_SHAP_mean"]), class_proba(pipe, row["text"]), prediction_cache), axis=1)
ensemble_eval["EXAI_LIME_SHAP_mean_sufficiency"] = ensemble_eval.progress_apply(lambda row: sufficiency(tokenizer, pipe, row[model_pred_col], eval(row["EnsembleXAI_LIME_SHAP_mean"]), class_proba(pipe, row["text"]), prediction_cache), axis=1)
ensemble_eval["EXAI_LIME_SHAP_mean_corr_loo"] = ensemble_eval.progress_apply(lambda row: correlation_leave_one_out(tokenizer, pipe, row[model_pred_col], eval(row["EnsembleXAI_LIME_SHAP_mean"]), class_proba(pipe, row["text"]), prediction_cache), axis=1)
ensemble_eval["EXAI_LIME_SHAP_mean_ins_AUC"] = ensemble_eval.progress_apply(lambda row: insertion_auc(tokenizer, pipe, row[model_pred_col], eval(row["EnsembleXAI_LIME_SHAP_mean"]), prediction_cache), axis=1)
ensemble_eval["EXAI_LIME_SHAP_mean_del_AUC"] = ensemble_eval.progress_apply(lambda row: deletion_auc(tokenizer, pipe, row[model_pred_col], eval(row["EnsembleXAI_LIME_SHAP_mean"]), prediction_cache), axis=1)

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

In [29]:
ensemble_eval["EXAI_LIME_SHAP_mean_combined"] = ensemble_eval.apply(lambda row: combined_metric(row["EXAI_LIME_SHAP_mean_comprehensiveness"], row["EXAI_LIME_SHAP_mean_sufficiency"], row["EXAI_LIME_SHAP_mean_corr_loo"], row["EXAI_LIME_SHAP_mean_ins_AUC"], row["EXAI_LIME_SHAP_mean_del_AUC"]), axis=1)

In [30]:
ensemble_eval["EXAI_LIME_SHAP_median_comprehensiveness"] = ensemble_eval.progress_apply(lambda row: comprehensivness(tokenizer, pipe, row[model_pred_col], eval(row["EnsembleXAI_LIME_SHAP_median"]), class_proba(pipe, row["text"]), prediction_cache), axis=1)
ensemble_eval["EXAI_LIME_SHAP_median_sufficiency"] = ensemble_eval.progress_apply(lambda row: sufficiency(tokenizer, pipe, row[model_pred_col], eval(row["EnsembleXAI_LIME_SHAP_median"]), class_proba(pipe, row["text"]), prediction_cache), axis=1)
ensemble_eval["EXAI_LIME_SHAP_median_corr_loo"] = ensemble_eval.progress_apply(lambda row: correlation_leave_one_out(tokenizer, pipe, row[model_pred_col], eval(row["EnsembleXAI_LIME_SHAP_median"]), class_proba(pipe, row["text"]), prediction_cache), axis=1)
ensemble_eval["EXAI_LIME_SHAP_median_ins_AUC"] = ensemble_eval.progress_apply(lambda row: insertion_auc(tokenizer, pipe, row[model_pred_col], eval(row["EnsembleXAI_LIME_SHAP_median"]), prediction_cache), axis=1)
ensemble_eval["EXAI_LIME_SHAP_median_del_AUC"] = ensemble_eval.progress_apply(lambda row: deletion_auc(tokenizer, pipe, row[model_pred_col], eval(row["EnsembleXAI_LIME_SHAP_median"]), prediction_cache), axis=1)

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

In [31]:
ensemble_eval["EXAI_LIME_SHAP_median_combined"] = ensemble_eval.apply(lambda row: combined_metric(row["EXAI_LIME_SHAP_median_comprehensiveness"], row["EXAI_LIME_SHAP_median_sufficiency"], row["EXAI_LIME_SHAP_median_corr_loo"], row["EXAI_LIME_SHAP_median_ins_AUC"], row["EXAI_LIME_SHAP_median_del_AUC"]), axis=1)

In [32]:
ensemble_eval.to_csv("data/xai_eval/xai_eval_ensemble_" + model_pred_col + ".csv", index=False)